# 03 — Huấn luyện 4 model (+ Dummy baseline)

Cùng preprocessor, cùng split, cùng metric. GridSearchCV **chỉ trên train**. Test dùng một lần ở notebook 04.

| Họ | Model |
|---|---|
| Baseline | Dummy (mean) + Linear Regression |
| Cây | Decision Tree |
| Ensemble bagging | Random Forest |
| Kernel | SVR |

Export: `../models/model.joblib` + `schema.json` + `metadata.json`.

Trên Colab sau khi chạy xong:

```python
from google.colab import files
files.download("../models/model.joblib")
```


In [ ]:

import sys
import time
import traceback
import logging

from pathlib import Path
from datetime import datetime


# ============================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN
# ============================================================

ROOT = Path("/content/11_10123189_10123262_InsuranceCost")

SRC = ROOT / "ai-models" / "src"

AI_ROOT = ROOT / "ai-models"

MODELS_DIR = AI_ROOT / "models"

COMP_DIR = MODELS_DIR / "comparisons"

LOG_DIR = AI_ROOT / "logs"

LOG_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOG_DIR / "train_debug.log"


# ============================================================
# 2. CẤU HÌNH LOGGER
# ============================================================

logger = logging.getLogger("TRAIN_DEBUG")

logger.setLevel(logging.DEBUG)

# Không cho log truyền lên root logger,
# tránh một thông báo bị in ra nhiều lần.
logger.propagate = False

# Xóa handler cũ nếu chạy lại cell
for handler in logger.handlers[:]:
    handler.close()
    logger.removeHandler(handler)

formatter = logging.Formatter(
    "%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# Log ra màn hình Colab
console_handler = logging.StreamHandler(sys.stdout)

console_handler.setLevel(logging.DEBUG)

console_handler.setFormatter(formatter)

# Log ra file
file_handler = logging.FileHandler(
    LOG_FILE,
    mode="a",
    encoding="utf-8"
)

file_handler.setLevel(logging.DEBUG)

file_handler.setFormatter(formatter)

logger.addHandler(console_handler)

logger.addHandler(file_handler)


# ============================================================
# 3. BẮT ĐẦU DEBUG
# ============================================================

start_time = time.time()

logger.info("=" * 70)

logger.info("BẮT ĐẦU CHƯƠNG TRÌNH TRAIN")

logger.info(f"Thời gian: {datetime.now()}")

logger.info(f"Thư mục dự án: {ROOT}")

logger.info(f"Thư mục src: {SRC}")

logger.info(f"Thư mục lưu model: {MODELS_DIR}")

logger.info(f"Thư mục lưu model so sánh: {COMP_DIR}")

logger.info(f"File log: {LOG_FILE}")


try:

    # ========================================================
    # 4. KIỂM TRA ĐƯỜNG DẪN
    # ========================================================

    logger.info("[1/6] KIỂM TRA THƯ MỤC DỰ ÁN")

    if not ROOT.exists():
        raise FileNotFoundError(
            f"Không tìm thấy thư mục dự án: {ROOT}"
        )

    if not SRC.exists():
        raise FileNotFoundError(
            f"Không tìm thấy thư mục src: {SRC}"
        )

    train_file = SRC / "train.py"

    if not train_file.exists():
        raise FileNotFoundError(
            f"Không tìm thấy file train.py: {train_file}"
        )

    logger.info(f"Đã tìm thấy train.py: {train_file}")

    logger.info("Kiểm tra đường dẫn thành công")


    # ========================================================
    # 5. IMPORT MODULE TRAIN
    # ========================================================

    logger.info("[2/6] IMPORT MODULE TRAIN")

    # Đưa thư mục src vào Python path
    src_path = str(SRC)

    if src_path not in sys.path:
        sys.path.insert(0, src_path)

    # Import module
    import train

    logger.info("Import train.py thành công")

    # Kiểm tra hàm main()
    if not hasattr(train, "main"):
        raise AttributeError(
            "File train.py không có hàm main()"
        )

    logger.info("Đã tìm thấy hàm main()")


    # ========================================================
    # 6. CHẠY HUẤN LUYỆN
    # ========================================================

    logger.info("[3/6] BẮT ĐẦU HUẤN LUYỆN MODEL")

    train_start = time.time()

    train.main()

    train_elapsed = time.time() - train_start

    logger.info(
        f"Huấn luyện hoàn tất sau {train_elapsed:.2f} giây"
    )


    # ========================================================
    # 7. KIỂM TRA CÁC FILE ĐẦU RA
    # ========================================================

    logger.info("[4/6] KIỂM TRA FILE ĐẦU RA")

    # Đường dẫn đầu ra đúng theo train.py
    expected_files = [
        MODELS_DIR / "model.joblib",
        MODELS_DIR / "results.csv",
        MODELS_DIR / "metadata.json",
        MODELS_DIR / "schema.json",
    ]

    all_files_exist = True

    for file_path in expected_files:

        if file_path.exists():

            size_kb = file_path.stat().st_size / 1024

            logger.info(
                f"[OK] {file_path.relative_to(ROOT)} "
                f"({size_kb:.2f} KB)"
            )

        else:

            all_files_exist = False

            logger.error(
                f"[THIẾU] Không tìm thấy: "
                f"{file_path.relative_to(ROOT)}"
            )


    # ========================================================
    # 8. KIỂM TRA CÁC MODEL SO SÁNH
    # ========================================================

    logger.info("[5/6] KIỂM TRA CÁC MODEL SO SÁNH")

    comparison_models = [
        COMP_DIR / "dummy_mean.pkl",
        COMP_DIR / "linear_regression.pkl",
        COMP_DIR / "decision_tree.pkl",
        COMP_DIR / "random_forest.pkl",
        COMP_DIR / "svr.pkl",
    ]

    for model_path in comparison_models:

        if model_path.exists():

            size_kb = model_path.stat().st_size / 1024

            logger.info(
                f"[OK] {model_path.relative_to(ROOT)} "
                f"({size_kb:.2f} KB)"
            )

        else:

            logger.warning(
                f"[THIẾU] {model_path.relative_to(ROOT)}"
            )


    # ========================================================
    # 9. TỔNG KẾT
    # ========================================================

    logger.info("[6/6] TỔNG KẾT")

    elapsed = time.time() - start_time

    if all_files_exist:

        logger.info(
            "KIỂM TRA THÀNH CÔNG: "
            "Tất cả file đầu ra chính đã tồn tại."
        )

    else:

        logger.warning(
            "Huấn luyện đã chạy xong nhưng "
            "còn thiếu một hoặc nhiều file đầu ra."
        )

    logger.info(
        f"Tổng thời gian chạy: {elapsed:.2f} giây"
    )

    logger.info(
        f"File log được lưu tại: {LOG_FILE}"
    )


except Exception as e:

    logger.error(
        f"CHƯƠNG TRÌNH THẤT BẠI: {e}"
    )

    logger.error(
        traceback.format_exc()
    )

    raise


finally:

    logger.info("=" * 70)

    # Đảm bảo log được ghi xuống file
    for handler in logger.handlers:
        handler.flush()

File `model.joblib` là **pipeline đầy đủ** (ColumnTransformer + estimator). Đưa vào `ai-models/models/` rồi `docker compose up --build ai-service` — model nạp ngay khi container start.
